# Single URL Product Scrape

Use this notebook to debug one product URL and inspect the artifact files. The scraper does not search; it only uses the supplied URL and optional product/context fields.

In [ ]:
from pathlib import Path
import sys, json

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_PATH:', SRC_PATH)
print('SRC exists:', SRC_PATH.exists())

## Optional: clear cached package modules

Run this cell after replacing files in the repo without restarting the kernel.

In [ ]:
import sys
for name in list(sys.modules):
    if name.startswith('product_scraping_agent'):
        del sys.modules[name]

from product_scraping_agent import ProductScrapingAgent, ScrapeRequest
print('Imported ProductScrapingAgent')

## Configure one scrape request

`requested_*` is the business/target context. `source_*` is the actual URL you are scraping. If the URL is a fallback source, set `source_url_role='global_fallback'` or another appropriate role.

In [ ]:
request = ScrapeRequest(
    product_url='https://www.amazon.com/Barbie-Collectible-All-Denim-Signature-Underwear/dp/B0BHFD7BPM',
    main_text='THE MOVIE 2023 KEN DOLL WEARING DENIM SET',
    ean='194735174539',

    # Original business/requested context
    requested_retailer_name='Mercado Libre',
    requested_country_code='CO',

    # Actual supplied URL/source context
    source_retailer_name='Amazon',
    source_country_code='US',
    source_url_role='global_fallback',

    output_root=PROJECT_ROOT / 'data' / 'scraped',
    max_images=30,
    vision_max=12,
    max_agent_iterations=2,
    write_raw_debug=False,
)
request

## Run scrape

In [ ]:
agent = ProductScrapingAgent()
result = await agent.scrape(request)
result.model_dump()

## Quick diagnostics

In [ ]:
print('Success:', result.success)
print('Quality:', result.artifact_quality)
print('Quality score:', result.quality_score)
print('Manual review:', result.requires_manual_review)
print('Access:', result.access_status, '| Browser visible:', result.browser_visible)
print('Product details recovered:', result.product_details_recovered)
print('Recovery status:', result.recovery_status)
print('Evidence axes:', result.evidence_axes_used)
print('Source alignment:', result.source_alignment.alignment_status)
print('Source claim scope:', result.source_alignment.source_specific_claim_scope)
print('Image candidates:', result.image_candidate_count)
print('Final clean product images:', result.final_image_count)
print('Output dir:', result.output_dir)
print('Error:', result.error)

## Read reports

In [ ]:
def read_json(path):
    path = Path(path) if path else None
    return json.loads(path.read_text(encoding='utf-8')) if path and path.exists() else {}

quality = read_json(result.quality_report_json_path)
alignment = read_json(result.source_alignment_report_json_path)
recovery = read_json(result.evidence_recovery_report_json_path)

print('QUALITY REPORT')
print(json.dumps(quality, indent=2, ensure_ascii=False))
print('\nSOURCE ALIGNMENT REPORT')
print(json.dumps(alignment, indent=2, ensure_ascii=False))
print('\nEVIDENCE RECOVERY REPORT')
print(json.dumps(recovery, indent=2, ensure_ascii=False))

## Inspect clean image folder

In [ ]:
images_dir = Path(result.output_dir) / 'images' if result.output_dir else None
if images_dir and images_dir.exists():
    files = sorted(p.name for p in images_dir.iterdir() if p.is_file())
else:
    files = []

print('Final clean image files:', len(files))
for f in files[:20]:
    print('-', f)

## Open business-readable artifacts

In [ ]:
for label, path in [
    ('product_evidence.md', result.product_evidence_md_path),
    ('claims.md', result.claims_md_path),
    ('vision.md', result.vision_md_path),
    ('source.md', result.source_md_path),
]:
    print('\n' + '='*80)
    print(label, path)
    print('='*80)
    if path and Path(path).exists():
        print(Path(path).read_text(encoding='utf-8')[:5000])
    else:
        print('(missing)')